In [ ]:
from datetime import datetime
from getpass import getpass

rdm_url = 'http://localhost:5001/'
idp_name_1 = 'FakeCAS'
idp_username_1 = None
idp_password_1 = None
project_count = 10
project_name_prefix = None
workflow_name = 'TEST-TEMPLATE-202512251608'
start_form_content = 'Test request content'
review_result = '承認'
review_comment = 'Approved by automated test'
default_result_path = None
close_on_fail = False
transition_timeout = 30000
browser_type = 'chromium'
ignore_https_errors = False

In [ ]:
import tempfile

if idp_username_1 is None:
    idp_username_1 = input(prompt=f'Username for {idp_name_1}')
if idp_password_1 is None:
    idp_password_1 = getpass(prompt=f'Password for {idp_username_1}@{idp_name_1}')

if project_name_prefix is None:
    project_name_prefix = datetime.now().strftime('TEST-WF-BATCH-%Y%m%d%H%M')
if workflow_name is None:
    workflow_name = input(prompt='Workflow template name')

project_names = [f'{project_name_prefix}-{i+1:02d}' for i in range(project_count)]
project_urls = []
print(f'Projects to create: {project_names}')

# ワークフローの一括実行

- サブシステム名: アドオン
- ページ/アドオン: Workflow
- 機能分類: ワークフロー一括実行
- シナリオ名: 複数プロジェクトでのワークフロー開始と承認
- 用意するテストデータ: アカウント、ワークフローテンプレート名
- 事前条件: テンプレート登録が完了していること

In [ ]:
work_dir = tempfile.mkdtemp()
if default_result_path is None:
    default_result_path = work_dir
work_dir

In [ ]:
import asyncio
import importlib
import pandas as pd

import scripts.playwright
importlib.reload(scripts.playwright)

from scripts.playwright import *
from scripts import grdm

await init_pw_context(close_on_fail=close_on_fail, last_path=default_result_path, ignore_https_errors=ignore_https_errors)

## 新規タブを開き、『同意する』をクリックする

GRDMトップページが表示されること

In [ ]:
async def _step(page):
    await page.goto(rdm_url)
    await page.locator('//button[text() = "同意する"]').click()
    await expect(page.locator('//button[text() = "同意する"]')).to_have_count(0, timeout=500)

await run_pw(_step, new_page=True)

## 『ユーザー名』『パスワード』を入力し『ログイン』をクリックする

GRDMダッシュボードが表示されること

In [ ]:
async def _step(page):
    await grdm.login(page, idp_name_1, idp_username_1, idp_password_1, transition_timeout=transition_timeout)
    await grdm.expect_dashboard(page, transition_timeout=transition_timeout)

await run_pw(_step)

## 各プロジェクトについて『新規プロジェクト作成』→『アドオン』→『Workflow有効にする』→『確認』→『テンプレート選択』→『有効化』→『権限を付与して有効化』をクリックする

全プロジェクト（project_count個）でWorkflowテンプレートが有効化されること

In [ ]:
async def _step(page):
    for idx, project_name in enumerate(project_names):
        print(f'[{idx+1}/{len(project_names)}] Setting up project: {project_name}')
        
        # ダッシュボードに戻る
        await page.goto(rdm_url)
        await grdm.expect_dashboard(page, transition_timeout=transition_timeout)
        
        # 『新規プロジェクト作成』をクリック
        await grdm.ensure_project_exists(page, project_name, transition_timeout=transition_timeout)
        
        # プロジェクトをクリック
        await page.locator(f'//*[@data-test-dashboard-item-title and text() = "{project_name}"]').click()
        await expect(page.locator('//span[@id = "nodeTitleEditable"]')).to_be_visible(timeout=transition_timeout)
        
        # プロジェクトURLを収集
        project_urls.append(page.url)
        
        # 『アドオン』をクリック
        await page.locator('//a[text() = "アドオン"]').click()
        await expect(page.locator('//h3[text() = "アドオンを選択"]')).to_be_visible(timeout=transition_timeout)
        
        # 『Workflow有効にする』をクリック
        await page.locator('//div[@full_name = "Workflow"]//a[text() = "有効にする"]').click()
        await expect(page.locator('//button[@data-bb-handler = "confirm"]')).to_be_visible(timeout=transition_timeout)
        
        # 『確認』をクリック
        await page.locator('//button[@data-bb-handler = "confirm"]').click()
        await expect(page.locator('#activate-workflow-select')).to_be_visible(timeout=transition_timeout)
        
        # 『テンプレート』を選択
        await page.locator('#activate-workflow-select').scroll_into_view_if_needed()
        option = page.locator(f'#activate-workflow-select option:has-text("{workflow_name}")')
        value = await option.get_attribute('value')
        await page.locator('#activate-workflow-select').select_option(value=value)
        
        # 『有効化』をクリック
        await page.locator('#activationsPanel').locator('//button[.//span[contains(text(), "有効化")]]').click()
        await expect(page.locator("#activateTokenPermissionModal").locator('//h4[contains(text(), "ワークフロー権限の付与")]')).to_be_visible(timeout=transition_timeout)

        # 『権限を付与して有効化』をクリック
        await page.locator('//button[contains(text(), "権限を付与して有効化")]').click()
        await expect(page.locator('#activationsPanel').locator(f'//td//strong[text()="{workflow_name}"]')).to_be_visible(timeout=transition_timeout)
        
        print(f'  -> Completed: {project_name} ({project_urls[-1]})')
    
    print(f'\nPhase 1 completed: {len(project_names)} projects prepared')

await run_pw(_step)

## 各プロジェクトについて『ワークフローリンク』→『内容を確認しました』→『申請内容』入力→『ワークフローを開始』をクリックする

全プロジェクトでワークフローが開始され、タスク一覧に『開く』ボタンが表示されること

In [ ]:
async def _step(page):
    for idx, project_url in enumerate(project_urls):
        project_name = project_names[idx]
        print(f'[{idx+1}/{len(project_urls)}] Starting workflow for: {project_name}')
        
        # プロジェクトURLに直接遷移
        await page.goto(project_url)
        await expect(page.locator('//span[@id = "nodeTitleEditable"]')).to_have_text(project_name, timeout=transition_timeout)
        await expect(page.locator('#workflow-dashboard')).to_be_visible(timeout=transition_timeout)
        
        # 『ワークフローリンク』をクリック
        await page.locator(f'#workflow-dashboard').locator(f'//a[contains(@class, "btn-primary") and contains(text(), "{workflow_name}")]').click()
        await expect(page.locator('#workflow-field-confirm_checkbox')).to_be_visible(timeout=transition_timeout)
        await asyncio.sleep(1)
        
        # 『内容を確認しました』をチェック
        await page.locator('#workflow-field-confirm_checkbox').check()
        
        # 『申請内容』に入力
        await page.locator('#workflow-field-request_content').fill(f'{start_form_content} - {project_name}')
        
        # 『ワークフローを開始』をクリック
        await page.locator('//button[@data-test-workflow-start-button]').click()
        await expect(page.locator('//*[@data-test-workflow-submit-success]')).to_be_visible(timeout=transition_timeout)
        
        print(f'  -> Workflow started: {project_name}')
    
    print(f'\nPhase 2 completed: {len(project_urls)} workflows started')

await run_pw(_step)

## 各プロジェクトについてコメントボタンをクリックし通知コメントを確認する

全プロジェクトで申請内容を含む通知コメントが表示されること

In [ ]:
async def _step(page):
    # タイマー（PT1S）完了を待つ
    await asyncio.sleep(2)
    
    for idx, project_url in enumerate(project_urls):
        project_name = project_names[idx]
        print(f'[{idx+1}/{len(project_urls)}] Checking comment for: {project_name}')
        
        # プロジェクトURLに直接遷移
        await page.goto(project_url)
        await expect(page.locator('//span[@id = "nodeTitleEditable"]')).to_have_text(project_name, timeout=transition_timeout)
        
        # コメントボタンをクリック
        await expect(page.locator('.fa-comments-o')).to_be_visible(timeout=transition_timeout)
        await page.locator('.fa-comments-o').click()
        
        # 申請内容を含むコメントが表示されることを確認
        expected_content = f'{start_form_content} - {project_name}'
        await expect(page.locator(f'//*[contains(text(), "{expected_content}")]')).to_be_visible(timeout=transition_timeout)
        
        print(f'  -> Comment verified: {project_name}')
    
    print(f'\nComment verification completed: {len(project_urls)} projects verified')

await run_pw(_step)

## 各プロジェクトについて『開く』→『結果』選択→『コメント』入力→『送信』をクリックする

全プロジェクトでワークフローが完了すること

In [ ]:
async def _step(page):
    for idx, project_url in enumerate(project_urls):
        project_name = project_names[idx]
        print(f'[{idx+1}/{len(project_urls)}] Completing task for: {project_name}')
        
        # プロジェクトURLに直接遷移
        await page.goto(project_url)
        await expect(page.locator('//span[@id = "nodeTitleEditable"]')).to_have_text(project_name, timeout=transition_timeout)
        await expect(page.locator('#workflow-dashboard')).to_be_visible(timeout=transition_timeout)
        
        # 『開く』をクリック
        await page.locator('//button[contains(@data-bind, "openTaskInWorkflowPage")]').click()
        await expect(page.locator('#workflow-field-approval_result')).to_be_visible(timeout=transition_timeout)
        
        # 『結果』を選択
        option = page.locator(f'#workflow-field-approval_result option:has-text("{review_result}")')
        value = await option.get_attribute('value')
        await page.locator('#workflow-field-approval_result').select_option(value=value)
        
        # 『コメント』に入力
        await page.locator('#workflow-field-review_comment').fill(f'{review_comment} - {project_name}')
        
        # 『送信』をクリック
        await page.locator('//button[@data-test-task-dialog-submit]').click()
        await expect(page.locator('//button[@data-test-task-dialog-submit]')).not_to_be_visible(timeout=transition_timeout)
        await asyncio.sleep(1)
        
        print(f'  -> Task completed: {project_name}')
    
    print(f'\nPhase 3 completed: {len(project_urls)} tasks completed')

await run_pw(_step)

## 後始末

In [ ]:
await finish_pw_context(screenshot=False, last_path=default_result_path)

In [ ]:
!rm -fr {work_dir}